# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data Contract - Plain Words (Part 1/2)

1. **Unit of Analysis (Grain):**
   - **One row = one content item (page) per client** (uniquely identified by `content_hash_id` and `client_hash_id`).
   - For the daily fact table `fact_content_daily_performance`, the grain is **one row per report_date x client x content item** (a page-day).
2. **Tables to be Used:**
   - `dim_clients` (client metadata, tracking starts)
   - `dim_content` (content metadata)
   - `fact_content_daily_performance` (daily GSC and GA4 metrics, partitioned by month)
3. **Time Window:**
   - For the Capstone model, we will use a **90-day trailing feature window** to predict a **30-day future outcome window**.
   - For the verification queries in this notebook, we focus on a mid-panel month slice: **March 2026** (`month=2026-03`), which we split into a 15-day feature window (`report_date <= '2026-03-15'`) and a 16-day outcome window (`report_date > '2026-03-15'`).

In [3]:
# Print out the data contract definitions for Section 1
print("Unit of Analysis: One content item (page) per client")
print("Tables Used: dim_clients, dim_content, fact_content_daily_performance")
print("Time Window: Trailing 90-day feature window -> Future 30-day outcome window")


Unit of Analysis: One content item (page) per client
Tables Used: dim_clients, dim_content, fact_content_daily_performance
Time Window: Trailing 90-day feature window -> Future 30-day outcome window


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Data Contract - Plain Words (Part 2/2)

4. **Target or Proxy to Predict/Rank:**
   - We will predict `is_declining`, a binary label where `1` indicates organic visibility decline (defined as a $>20\%$ drop in GSC impressions in the outcome window compared to the feature window) and `0` otherwise.
5. **Deliberately Excluded Column (leakage prevention):**
   - **`trend_direction` and `trend_pct`:** These columns directly encode the target outcome (whether visibility declined or grew). Using them as features would result in 100% target leakage, causing the model to learn the rule instead of finding real signals.

### Field Classification

- **Features (Knowable before the prediction moment):**
  - `imp_start` (historical GSC impressions)
  - `clk_start` (historical GSC clicks)
  - `pos_start` (historical average position)
  - `ga4_sessions_start` (historical total GA4 sessions)
  - `has_ga4` (boolean check of whether GA4 tracking was available)
- **Label / Proxy:**
  - `is_declining` (derived from impressions in the future outcome window)
- **Context (IDs, splits, grouping - never for the model to learn):**
  - `content_hash_id` (pseudonymous page ID)
  - `client_hash_id` (pseudonymous client ID, used for client-holdout validation)
- **Excluded (leakage or private):**
  - `trend_direction` & `trend_pct` (target leakage)
  - `imp_end` (future outcome data, only used to construct the label)

In [5]:
# Print confirmation of the sorted fields
print("Features: imp_start, clk_start, pos_start, ga4_sessions_start, has_ga4")
print("Label: is_declining")
print("Context: content_hash_id, client_hash_id")
print("Excluded: trend_direction, trend_pct, imp_end")


Features: imp_start, clk_start, pos_start, ga4_sessions_start, has_ga4
Label: is_declining
Context: content_hash_id, client_hash_id
Excluded: trend_direction, trend_pct, imp_end


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Below, we execute three queries to verify our facts:
1. **The Grain:** Verify that report_date + client_hash_id + content_hash_id is unique in the raw monthly slice.
2. **Count and Date Span:** Verify the total row count and date boundaries for March 2026.
3. **Availability:** Show the count of rows that have GA4 tracking available.

We then build our 5-feature vector, train an honest classifier, and run the **leakage trap** experiment where we deliberately introduce a leaked column `trend_pct_leaked` and show the score jump to 1.0, before deleting it.

In [7]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

# 1. Connect to DuckDB and set up dataset path
con = duckdb.connect()
parquet_path = "data/fact_content_daily_performance/month=2026-03/data_0.parquet"

print("--- FACT 1: Verify the Grain ---")
# Check that report_date + client_hash_id + content_hash_id is unique in the raw daily fact table
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS cnt
    FROM read_parquet('{parquet_path}')
    GROUP BY 1, 2, 3
    HAVING cnt > 1
    LIMIT 5
""").df()
print(f"Rows violating daily grain (should be 0 rows): {len(grain_check)}")

print("\n--- FACT 2: Row Count and Date Span ---")
count_span = con.sql(f"""
    SELECT COUNT(*) AS total_rows, 
           MIN(report_date) AS min_date, 
           MAX(report_date) AS max_date
    FROM read_parquet('{parquet_path}')
""").df()
print(count_span)

print("\n--- FACT 3: Availability of GA4 Tracking ---")
# Count total rows and GA4 available rows in March 2026
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(CASE WHEN ga4_data_available = TRUE THEN 1 END) AS ga4_available_rows,
           AVG(CASE WHEN ga4_data_available = TRUE THEN 1.0 ELSE 0.0 END) * 100 AS ga4_available_pct
    FROM read_parquet('{parquet_path}')
""").df()
print(availability)

print("\n--- BUILD FEATURE VECTOR (5 Features Max) ---")
# Aggregate the month into feature (days 1-15) and outcome (days 16-31) windows
agg_query = f"""
    WITH feature_window AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_start,
               SUM(gsc_clicks) AS clk_start,
               COALESCE(AVG(gsc_avg_position), 0) AS pos_start,
               COALESCE(SUM(ga4_sessions), 0) AS ga4_sessions_start,
               COALESCE(MAX(CAST(ga4_data_available AS INT)), 0) AS has_ga4
        FROM read_parquet('{parquet_path}')
        WHERE report_date <= '2026-03-15'
        GROUP BY 1, 2
    ),
    outcome_window AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_end
        FROM read_parquet('{parquet_path}')
        WHERE report_date > '2026-03-15'
        GROUP BY 1, 2
    )
    SELECT f.client_hash_id, f.content_hash_id,
           f.imp_start, f.clk_start, f.pos_start, f.ga4_sessions_start, f.has_ga4,
           COALESCE(o.imp_end, 0) AS imp_end
    FROM feature_window f
    LEFT JOIN outcome_window o ON f.client_hash_id = o.client_hash_id AND f.content_hash_id = o.content_hash_id
    WHERE f.imp_start >= 50
"""
df = con.sql(agg_query).df()
print(f"Aggregated page-level dataframe shape: {df.shape}")

# Define target label
df["is_declining"] = (df["imp_end"] < 0.8 * df["imp_start"]).astype(int)

# 2. Honest Model Evaluation
X_honest = df[['imp_start', 'clk_start', 'pos_start', 'ga4_sessions_start', 'has_ga4']]
y = df['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
model_honest.fit(X_tr, y_tr)
auc_honest = roc_auc_score(y_te, model_honest.predict_proba(X_te)[:, 1])
print(f"Honest Model ROC-AUC: {auc_honest:.4f}")

# 3. The Leakage Trap
print("\n--- THE LEAKAGE TRAP (Target Leakage Column Added) ---")
df["trend_pct_leaked"] = (df["imp_end"].fillna(0) - df["imp_start"]) / df["imp_start"]
X_leaked = df[['imp_start', 'clk_start', 'pos_start', 'ga4_sessions_start', 'has_ga4', 'trend_pct_leaked']]

X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.25, random_state=42, stratify=y)
model_leaked = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
model_leaked.fit(X_tr_l, y_tr)
auc_leaked = roc_auc_score(y_te, model_leaked.predict_proba(X_te_l)[:, 1])
print(f"Leaked Model (The Trap) ROC-AUC: {auc_leaked:.4f}")

# 4. Remove the Trap Column
print("\n--- CLEANUP: Removing Leaked Column ---")
df = df.drop(columns=["trend_pct_leaked"])
print("Successfully removed leaked column 'trend_pct_leaked'.")


--- FACT 1: Verify the Grain ---
Rows violating daily grain (should be 0 rows): 0

--- FACT 2: Row Count and Date Span ---
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

--- FACT 3: Availability of GA4 Tracking ---
   total_rows  ga4_available_rows  ga4_available_pct
0     9841378              413966           4.206382

--- BUILD FEATURE VECTOR (5 Features Max) ---
Aggregated page-level dataframe shape: (92548, 8)
Honest Model ROC-AUC: 0.6381

--- THE LEAKAGE TRAP (Target Leakage Column Added) ---
Leaked Model (The Trap) ROC-AUC: 1.0000

--- CLEANUP: Removing Leaked Column ---
Successfully removed leaked column 'trend_pct_leaked'.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Key Data Limits & Caveats:

1. **Unbalanced History:**
   - History depth differs wildly per client (`dim_clients.gsc_data_start`). A global calendar-based time window will result in missing historical records for newer clients. We must establish relative time windows per client or filter out clients with insufficient history.
2. **GSC-Only Early Rows:**
   - Rows prior to a client's `ga4_data_start` have their GA4 columns zero-filled with `ga4_data_available = FALSE`. We must filter on `ga4_data_available = TRUE` before using engagement signals, or use flags, to avoid treating "no tracking yet" as "zero sessions/engagement".
3. **Overlapping Windows (Leakage):**
   - The query table (`fact_content_query_90d`) contains GSC statistics over a fixed trailing 90-day window. If our target label window overlaps with this 90-day window, the queries will leak the future outcome. We must ensure our feature and outcome windows do not overlap.
4. **Correlation vs. Causation:**
   - This observational data cannot prove that a content refresh *caused* a recovery. It only supports decision-making by prioritizing pages that show signs of organic decline.

In [9]:
# Print summary of data limitations
print("Data Limit 1: Unbalanced panel history per client")
print("Data Limit 2: GSC-only early rows (GA4 zero-filled when ga4_data_available is False)")
print("Data Limit 3: GSC Query table overlap leakage risk")


Data Limit 1: Unbalanced panel history per client
Data Limit 2: GSC-only early rows (GA4 zero-filled when ga4_data_available is False)
Data Limit 3: GSC Query table overlap leakage risk


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.